# 03 — Detector characterisation: what gain law did the GR predictor learn?

**No training, no dataset labels needed** (except two music excerpts for the
invariance checks). The `DetectorGRLSTM` is treated like a *device on a test bench*
and characterised with the classical signal-based methodology for dynamic range
compressors (Eichas & Zölzer-style static curve + ballistics probes, as also used in
the Optical-DRC reference): synthetic tones and level steps go in, the predicted GR
trajectory comes out.

This turns "test GR MAE 0.268 dB" into *interpretable claims*:

1. **Static compression curve** — steady-state GR vs input level per setting →
   the learned threshold/ratio/knee, compared against the knob labels.
2. **Knob interpolation** — sweeps through knob values *not in the 10 training
   settings*: a learned gain law should vary smoothly and monotonically; memorised
   settings would show kinks at the training grid.
3. **Ballistics** — level-step responses → fitted attack/release time constants vs
   the labelled knob values, plus programme-dependence (step-size) checks.
4. **Invariance audit** — the frontend is timbre/phase-blind *by construction*;
   here we measure the residual end-to-end sensitivity (incl. the LSTM), and the
   *level equivariance* that a compressor must have instead.
5. **Learned detector τ / kernels** — the model's one interpretable internal knob.

> Absolute calibration caveat: the SSL's threshold is calibrated in device units,
> our probes in dBFS — so extracted thresholds are compared *relatively* (spacing,
> ordering, linearity vs knob), not absolutely. Runs in ~2 min on CPU.

In [ ]:
# -- 0. Setup -------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from scipy.signal import lfilter

from exp_common import (
    SR, ensure_eval_out, fit_t63, knobs, level_step, load_detector, load_pair,
    load_split, noise, pairs_from_keys, params_for, rms_dbfs, sine,
)

SELECTED_DETECTOR_RUN = "lstm_gr_20260702_174039_lstm_detector_gr"
DET, HP, RUN_DIR = load_detector(SELECTED_DETECTOR_RUN)
SPLIT = load_split(RUN_DIR)
HOP = int(DET.hop_size)
OUT = ensure_eval_out()

SETTINGS = SPLIT.all_settings
print(f"{len(SETTINGS)} dataset settings; frame rate {SR/HOP:.1f} Hz")


@torch.no_grad()
def probe_gr(x, k):
    # Single cold-start forward (probes are short); frame-rate GR + time axis.
    gr = DET(x.unsqueeze(0).float(), k).squeeze().numpy()
    t = (np.arange(len(gr)) + 1) * HOP / SR
    return t, gr


def steady_gr(x, k, tail_sec=1.5):
    t, gr = probe_gr(x, k)
    return float(gr[-int(tail_sec * SR / HOP):].mean())

In [ ]:
# -- 1. Static compression curve per dataset setting ----------------------
# 4 s tone bursts (1 kHz), peak level swept -40..0 dBFS; steady-state GR from
# the final 1.5 s (past the warmup ambiguity and any release transient).

LEVELS = np.arange(-40, 1, 2.0)
F0 = 1000.0

static = {}
for setting in SETTINGS:
    k = params_for(setting)
    static[setting] = np.array([steady_gr(sine(F0, 4.0, lv), k) for lv in LEVELS])
    print(f"{setting:45s} GR @ 0 dBFS = {static[setting][-1]:6.2f} dB")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13.5, 5))
cmap = plt.cm.viridis(np.linspace(0, 1, len(SETTINGS)))
for c, setting in zip(cmap, SETTINGS):
    ax1.plot(LEVELS, static[setting], marker=".", ms=4, color=c, label=setting[10:])
    ax2.plot(LEVELS, LEVELS + static[setting], marker=".", ms=4, color=c)
ax1.set_xlabel("input peak level (dBFS)"); ax1.set_ylabel("steady-state GR (dB)")
ax1.set_title("Learned static GR curves"); ax1.grid(alpha=0.3)
ax1.legend(fontsize=6, loc="lower left")
ax2.plot(LEVELS, LEVELS, "k--", lw=0.8, label="unity")
ax2.set_xlabel("input level (dB)"); ax2.set_ylabel("output level (dB)")
ax2.set_title("Implied I/O characteristic"); ax2.grid(alpha=0.3); ax2.legend()
fig.tight_layout()
fig.savefig(OUT / "03_static_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# -- 2. Gain-law extraction: effective ratio and threshold ----------------
# Fit out = a*in + b on the compressed region (points with GR < -1 dB):
# ratio_hat = 1/a; threshold_hat = b/(1-a) (intersection with the unity line).
# Compare against knob labels - RELATIVE comparison (see calibration caveat).

from splits import setting_threshold, DIFFSSL_PARAM_RANGES  # 06_output copy
from src.dsp import parse_settings_from_folder_name

rows = []
for setting in SETTINGS:
    gr = static[setting]
    out_lv = LEVELS + gr
    mask = gr < -1.0
    parsed = parse_settings_from_folder_name(setting)
    if mask.sum() >= 3:
        a, b = np.polyfit(LEVELS[mask], out_lv[mask], 1)
        ratio_hat = 1.0 / a if a > 1e-3 else np.inf
        thr_hat = b / (1.0 - a) if abs(1.0 - a) > 1e-6 else np.nan
    else:
        ratio_hat, thr_hat = np.nan, np.nan
    rows.append({"setting": setting,
                 "thr (label)": float(parsed["threshold"]),
                 "thr̂ (dBFS)": thr_hat,
                 "ratio (label)": float(parsed["ratio"]),
                 "ratiô": ratio_hat,
                 "compressed pts": int(mask.sum())})
law_df = pd.DataFrame(rows).sort_values("thr (label)")
law_df.to_csv(OUT / "03_gain_law_extraction.csv", index=False)
display(law_df.round(2))

ok = law_df.dropna()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))
ax1.scatter(ok["thr (label)"], ok["thr̂ (dBFS)"], c="#d62728")
a1, b1 = np.polyfit(ok["thr (label)"], ok["thr̂ (dBFS)"], 1)
xs = np.array([ok["thr (label)"].min(), ok["thr (label)"].max()])
ax1.plot(xs, a1 * xs + b1, "k--", lw=0.8, label=f"fit: slope {a1:.2f}")
ax1.set_xlabel("threshold knob (device dB)"); ax1.set_ylabel("extracted threshold (dBFS)")
ax1.grid(alpha=0.3); ax1.legend(); ax1.set_title("Threshold: extracted vs label (slope→1 = linear law)")
ax2.scatter(ok["ratio (label)"], ok["ratiô"], c="#d62728")
lim = max(ok["ratio (label)"].max(), ok["ratiô"].max()) * 1.1
ax2.plot([0, lim], [0, lim], "k--", lw=0.8)
ax2.set_xlabel("ratio knob"); ax2.set_ylabel("extracted ratio")
ax2.grid(alpha=0.3); ax2.set_title("Ratio: extracted vs label")
fig.tight_layout()
fig.savefig(OUT / "03_gain_law_extraction.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# -- 3. Knob interpolation: unseen knob values ----------------------------
# The 10 training settings sample the knob space sparsely. Sweep threshold and
# ratio through values NOT on that grid: a learned law -> smooth monotone
# family; memorisation -> kinks/plateaus between training values.

THR_SWEEP = np.arange(-16, 17, 4.0)          # training grid: {-12,-8,-4,0,4,8,12}
RATIO_SWEEP = [1.5, 2, 3, 4, 5, 6, 8, 10]    # training grid: {2,4,10}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13.5, 5))
cm1 = plt.cm.plasma(np.linspace(0, 0.9, len(THR_SWEEP)))
curves_thr = {}
for c, thr in zip(cm1, THR_SWEEP):
    k = knobs(thr, 3.0, 0.4, 4.0)
    curves_thr[thr] = np.array([steady_gr(sine(F0, 4.0, lv), k) for lv in LEVELS])
    ax1.plot(LEVELS, curves_thr[thr], color=c, marker=".", ms=3,
             label=f"thr {thr:+.0f}")
ax1.set_xlabel("input peak level (dBFS)"); ax1.set_ylabel("steady GR (dB)")
ax1.set_title("Threshold sweep (attack 3 ms, release 0.4 s, ratio 4)")
ax1.grid(alpha=0.3); ax1.legend(fontsize=7)

cm2 = plt.cm.plasma(np.linspace(0, 0.9, len(RATIO_SWEEP)))
curves_ratio = {}
for c, r in zip(cm2, RATIO_SWEEP):
    k = knobs(-8.0, 3.0, 0.4, r)
    curves_ratio[r] = np.array([steady_gr(sine(F0, 4.0, lv), k) for lv in LEVELS])
    ax2.plot(LEVELS, curves_ratio[r], color=c, marker=".", ms=3, label=f"ratio {r}")
ax2.set_xlabel("input peak level (dBFS)"); ax2.set_ylabel("steady GR (dB)")
ax2.set_title("Ratio sweep (threshold -8)")
ax2.grid(alpha=0.3); ax2.legend(fontsize=7)
fig.tight_layout()
fig.savefig(OUT / "03_knob_interpolation.png", dpi=150, bbox_inches="tight")
plt.show()

# monotonicity audit: GR at -6 dBFS as a function of each knob
gr_at = lambda curves, keys: [curves[k][list(LEVELS).index(-6.0)] for k in keys]
thr_vals, ratio_vals = gr_at(curves_thr, THR_SWEEP), gr_at(curves_ratio, RATIO_SWEEP)
print("GR @ -6 dBFS vs threshold:", np.round(thr_vals, 2),
      "\n  monotone non-decreasing:", bool(np.all(np.diff(thr_vals) >= -0.05)))
print("GR @ -6 dBFS vs ratio    :", np.round(ratio_vals, 2),
      "\n  monotone non-increasing:", bool(np.all(np.diff(ratio_vals) <= 0.05)))

In [ ]:
# -- 4. Ballistics: fitted attack / release constants vs knobs ------------
# Level steps -30->-10 dBFS (attack) and -10->-30 (release), 3 s per phase,
# t63 fitted on the GR trajectory after the step. Sweeps include values off
# the training grid (attack grid {1,3,10,30}, release grid {0.1,0.4,0.8}).

ATTACKS = [1.0, 2.0, 3.0, 5.0, 10.0, 20.0, 30.0]
RELEASES = [0.1, 0.2, 0.4, 0.6, 0.8, 1.2]


def step_t63(k, direction):
    if direction == "attack":
        x = level_step(F0, -30.0, -10.0, 3.0, 3.0)
    else:
        x = level_step(F0, -10.0, -30.0, 3.0, 3.0)
    t, gr = probe_gr(x, k)
    pre = gr[(t > 2.5) & (t < 3.0)].mean()
    post = gr[t > 5.5].mean()
    m = t > 3.0
    return fit_t63(t[m] - 3.0, gr[m], pre, post), (t, gr, pre, post)


att_fit = [step_t63(knobs(-8.0, a, 0.4, 4.0), "attack")[0] for a in ATTACKS]
rel_fit = [step_t63(knobs(-8.0, 3.0, r, 4.0), "release")[0] for r in RELEASES]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
axes[0].plot(ATTACKS, np.array(att_fit) * 1000, "o-", color="#d62728")
axes[0].plot(ATTACKS, ATTACKS, "k--", lw=0.8, label="y = x")
axes[0].set_xlabel("attack knob (ms)"); axes[0].set_ylabel("fitted t63 (ms)")
axes[0].set_title("Attack ballistics"); axes[0].grid(alpha=0.3); axes[0].legend()
axes[1].plot(RELEASES, rel_fit, "o-", color="#d62728")
axes[1].plot(RELEASES, RELEASES, "k--", lw=0.8, label="y = x")
axes[1].set_xlabel("release knob (s)"); axes[1].set_ylabel("fitted t63 (s)")
axes[1].set_title("Release ballistics"); axes[1].grid(alpha=0.3); axes[1].legend()

# programme dependence: release t63 vs step DEPTH at fixed knobs (SSL-style
# auto-release would make deeper steps recover differently)
DEPTHS = [-5.0, -10.0, -15.0, -20.0]
dep = []
for d in DEPTHS:
    x = level_step(F0, -10.0, -10.0 + d, 3.0, 3.0)   # step DOWN by |d|
    t, gr = probe_gr(x, knobs(-8.0, 3.0, 0.4, 4.0))
    pre = gr[(t > 2.5) & (t < 3.0)].mean()
    post = gr[t > 5.5].mean()
    m = t > 3.0
    dep.append(fit_t63(t[m] - 3.0, gr[m], pre, post))
axes[2].plot(np.abs(DEPTHS), dep, "o-", color="#9467bd")
axes[2].set_xlabel("release step depth (dB)"); axes[2].set_ylabel("fitted t63 (s)")
axes[2].set_title("Programme dependence of release"); axes[2].grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT / "03_ballistics.png", dpi=150, bbox_inches="tight")
plt.show()

# example trajectory
tau, (t, gr, pre, post) = step_t63(knobs(-8.0, 3.0, 0.4, 4.0), "attack")
plt.figure(figsize=(10, 3))
plt.plot(t, gr, lw=0.9); plt.axvline(3.0, color="k", lw=0.6, alpha=0.5)
plt.axhline(pre, color="gray", lw=0.6); plt.axhline(post, color="gray", lw=0.6)
plt.title(f"Attack step response (thr -8, atk 3 ms, rel 0.4 s, ratio 4) - t63 = {tau*1000:.1f} ms")
plt.xlabel("time (s)"); plt.ylabel("GR (dB)"); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# -- 5. Invariance audit ---------------------------------------------------
# (a) timbre: four signals with EQUAL RMS level but different spectra. The
#     detector frontend sees only frame energy, so predictions should collapse;
#     the measured spread is the residual timbre sensitivity of the whole model.
# (b) phase: an all-pass-filtered music excerpt (|H|=1 at all f) must map to
#     the same GR. (c) level equivariance: +-6 dB input shifts should move GR
#     along the static curve, not leave it unchanged.

k = knobs(-8.0, 3.0, 0.4, 4.0)

# (a) timbre
sigs = {
    "sine 1 kHz": sine(1000.0, 3.0, -15.0 + 3.01),     # peak -> approx -15 dB RMS
    "sine 200 Hz": sine(200.0, 3.0, -15.0 + 3.01),
    "white noise": noise(3.0, -15.0, "white"),
    "pink noise": noise(3.0, -15.0, "pink"),
}
fig, axes = plt.subplots(1, 3, figsize=(15.5, 4))
steady = {}
for name, x in sigs.items():
    t, gr = probe_gr(x, k)
    steady[name] = gr[t > 1.5].mean()
    axes[0].plot(t, gr, lw=0.9, label=f"{name} (RMS {rms_dbfs(x):.1f} dB)")
axes[0].set_title(f"Timbre invariance - steady spread "
                  f"{max(steady.values()) - min(steady.values()):.3f} dB")
axes[0].set_xlabel("time (s)"); axes[0].set_ylabel("GR (dB)")
axes[0].grid(alpha=0.3); axes[0].legend(fontsize=7)

# (b) phase: all-pass cascade on a validation-song excerpt
val_pairs = pairs_from_keys(SPLIT.val_pair_keys)
song, setting = val_pairs[0]
dry, _, _ = load_pair(setting, song, start_sec=30.0, duration_sec=20.0)
x = dry[0].numpy()
for a in (0.3, -0.5, 0.7, -0.2):                      # 1st-order allpasses, |H|=1
    x = lfilter([a, 1.0], [1.0, a], x)
dry_ap = torch.from_numpy(x.astype(np.float32)).unsqueeze(0)
t, gr0 = probe_gr(dry, params_for(setting))
_, gr1 = probe_gr(dry_ap, params_for(setting))
axes[1].plot(t, gr0, lw=0.8, label="original")
axes[1].plot(t, gr1, lw=0.8, alpha=0.75, label="all-passed")
axes[1].set_title(f"Phase invariance - MAE {np.abs(gr0 - gr1).mean():.4f} dB")
axes[1].set_xlabel("time (s)"); axes[1].grid(alpha=0.3); axes[1].legend(fontsize=8)

# (c) level equivariance on the same excerpt
for shift, color in ((-6.0, "#1f77b4"), (0.0, "k"), (6.0, "#d62728")):
    _, g = probe_gr(dry * 10 ** (shift / 20), params_for(setting))
    axes[2].plot(t, g, lw=0.8, color=color, alpha=0.8, label=f"{shift:+.0f} dB input")
axes[2].set_title("Level equivariance (must NOT be invariant)")
axes[2].set_xlabel("time (s)"); axes[2].grid(alpha=0.3); axes[2].legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT / "03_invariances.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"timbre steady-state GR spread : {max(steady.values()) - min(steady.values()):.3f} dB")
print(f"phase (all-pass) GR MAE       : {np.abs(gr0 - gr1).mean():.4f} dB")

In [ ]:
# -- 6. Learned detector time constants + effective kernels ----------------
taus = [float(v) for v in DET.detector.taus_ms]
print(f"detector tau init (ms): {HP['model']['detector_taus_ms_init']}")
print(f"detector tau learned  : {[round(v, 1) for v in taus]}")
print(f"label RMS window      : 1024 samples = {1024/SR*1000:.1f} ms (trailing)")

with torch.no_grad():
    kern = DET.detector.kernels().squeeze(1).numpy()     # [N, K] causal (flipped)
t_k = np.arange(kern.shape[-1])[::-1] * HOP / SR         # age of each tap (s)
plt.figure(figsize=(9, 3.5))
for i, kv in enumerate(kern):
    plt.plot(t_k, kv, lw=1.0, label=f"tau = {taus[i]:.1f} ms")
plt.xlabel("age (s)"); plt.ylabel("kernel weight")
plt.title("Learned detector impulse responses (energy-domain smoothing)")
plt.grid(alpha=0.3); plt.legend(); plt.xlim(0, 0.6)
plt.tight_layout()
plt.savefig(OUT / "03_detector_kernels.png", dpi=150, bbox_inches="tight")
plt.show()

## Reading the results

- **Static curves**: threshold ordering/spacing and ratio steepness should track the
  knobs. The regression slope of extracted-vs-label threshold near 1 means the model
  encodes the threshold *law*, not ten lookup entries. Settings that never compress
  at 0 dBFS (high thresholds) legitimately extract `ratiô ≈ 1`.
- **Interpolation sweeps**: smooth monotone families at unseen knob values are the
  strongest cheap evidence against setting memorisation — quote the monotonicity
  audit directly.
- **Ballistics**: fitted-vs-label scatter on the identity line = the LSTM learned
  attack/release *as functions of the knobs*. Deviations at 1 ms attack are expected:
  one frame is already 5.8 ms, the model's Nyquist for ballistics. The
  programme-dependence panel probes SSL auto-release-like behaviour a fixed
  one-pole ballistics model could not express.
- **Invariances**: the timbre spread bounds how much non-level information survives
  the frontend end-to-end (claim in `MODEL_DETECTOR_GR.md` §2.1); level equivariance
  is the sanity check that the collapse is not trivial (a constant predictor would
  also be "invariant").